# AgentCore MCP Runtime Test

## Prerequisites

Ensure you are authenticated with AWS (for example, run `aws sso login --profile YOUR_PROFILE` or configure your AWS credentials) before accessing your runtime.

In [ ]:
arn = "" # TODO enter the ARN of your AgentCoreRuntime
neo4j_user = "companies"
neo4j_password = "companies"

In [ ]:
!pip install botocore[crt]

from urllib.parse import quote
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client
from strands.tools.mcp import MCPClient
from strands import Agent
import base64
import boto3


In [ ]:
ENCODED_ARN = quote(arn, safe="")

# Build Basic auth header from user:password
credentials = base64.b64encode(f"{neo4j_user}:{neo4j_password}".encode()).decode()
auth_header = f"Basic {credentials}"
region = boto3.Session().region_name
mcp_client = MCPClient(lambda: aws_iam_streamablehttp_client(
    endpoint="https://bedrock-agentcore." + region + ".amazonaws.com/runtimes/" + ENCODED_ARN + "/invocations?qualifier=DEFAULT",
    aws_region=region,
    aws_service="bedrock-agentcore",
    headers={
        # Use the computed header instead of a hardcoded value
        "X-Amzn-Bedrock-AgentCore-Runtime-Custom-Authorization": auth_header,
    },
))

mcp_client.start()


In [ ]:
result = mcp_client.list_tools_sync()

for t in result:
    title = t.tool_name
    desc = t.tool_spec.get("description")
    print(f"- {title}: {desc}")

In [ ]:
result = mcp_client.call_tool_sync(
    "1", "get-schema",
)
print(result)


Close the MCP client

In [ ]:
mcp_client.stop(None, None, None)

Use the MCP via Agent

In [ ]:
agent = Agent(tools=[mcp_client])
print(agent("Give me the name of 3 Companies"))